In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Job parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")
if not table_name:
    print("  WARNING: table_name is empty")
if not landing_timestamp:
    print("  WARNING: landing_timestamp is empty")

In [0]:
#Read bronze batch
source_table = f"adb_rapollocare.bronze.{table_name}"
filter_condition = f"landing_timestamp='{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading batch from Bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent query :")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Rows read from bronze : {bronze_count}")
print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
if bronze_count == 0:
    print("  WARNING: No rows found in bronze for this landing_timestamp")

In [0]:
from pyspark.sql.functions import current_timestamp

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print("  Adding column: insert_timestamp = current_timestamp()")
print("  Adding column: update_timestamp = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_rapollocare.silver.{table_name}"
load_type_normalized = load_type.strip().upper() #load_type=string, strip()=remove leading and trailing spaces, upper()=convert to uppercase

print("=" * 80)
print("STEP 4: Writing to Silver")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Load type (raw)      : '{load_type}'")
print(f"  Load type (resolved) : '{load_type_normalized}'")

if load_type_normalized == "FULL":
    # Full load: replace target with the current batch
    print("  Strategy : FULL load -> overwrite target with current batch")
    print("  Equivalent query :")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS")
    print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
    print(f"    FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"  SUCCESS: {target_table} overwritten")

elif load_type_normalized == "APPEND":
    # Append: add the current batch to the target as-is
    print("  Strategy : APPEND -> add current batch to target as-is")
    print("  Equivalent query :")
    print(f"    INSERT INTO {target_table}")
    print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
    print(f"    FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"  SUCCESS: batch appended to {target_table}")

elif load_type_normalized == "MERGE":
    # Merge (upsert): update matching records, insert new ones, based on merge_keys
    print("  Strategy : MERGE (upsert) -> update matching rows, insert new rows")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Parsed merge keys : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")

    if not spark.catalog.tableExists(target_table):
        # First run: target does not exist yet, so create it from the current batch
        print(f"  Target {target_table} does not exist -> first run, creating it from current batch")
        print("  Equivalent query :")
        print(f"    CREATE TABLE {target_table} AS")
        print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
        print(f"    FROM {source_table} WHERE {filter_condition}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  SUCCESS: {target_table} created")
    else:
        # Ensure one source row per key to avoid multiple-match merge errors
        print(f"  Target {target_table} exists -> performing MERGE")
        source_df = updated_df.dropDuplicates(keys)
        dedup_count = source_df.count()
        print(f"  Source rows after de-duplication on {keys} : {dedup_count}")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update keep the original insert_timestamp; refresh everything else incl. update_timestamp
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        update_clause = ",\n        ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        print(f"  Merge condition : {merge_condition}")
        print("  Equivalent query :")
        print(f"    MERGE INTO {target_table} AS t")
        print(f"    USING (deduplicated batch from {source_table} WHERE {filter_condition}) AS s")
        print(f"    ON {merge_condition}")
        print("    WHEN MATCHED THEN UPDATE SET")
        print(f"        {update_clause}")
        print(f"    WHEN NOT MATCHED THEN INSERT ({insert_cols})")
        print(f"    VALUES ({insert_vals})")

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute()
        )

        # Report merge metrics from the Delta history
        last_op = DeltaTable.forName(spark, target_table).history(1).select("operation", "operationMetrics").collect()[0]
        metrics = last_op["operationMetrics"] or {}
        print(f"  SUCCESS: MERGE completed on {target_table}")
        print(f"    Rows updated  : {metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Rows inserted : {metrics.get('numTargetRowsInserted', 'n/a')}")
        print(f"    Rows deleted  : {metrics.get('numTargetRowsDeleted', 'n/a')}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("STEP 5: Completed")
print("=" * 80)
print(f"  Table           : {table_name}")
print(f"  Load type       : {load_type_normalized}")
print(f"  Records in batch: {record_count}")
print(f"  Returning '{record_count}' to the calling job via dbutils.notebook.exit")
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from adb_rapollocare.silver.patients